In [ ]:
import os
import time
import random
import json
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score
from thop import profile
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Import all routing modules (use the improved versions)
from routing_smoe import SMoELayer
from routing_micro import MICROMoELayer
from routing_expert_choice import ExpertChoiceMoELayer
from routing_adaptive import AdaptiveDynamicMoELayer
from routing_deepseek import DeepSeekMoELayer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Running on: {device}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f" Seed set to {seed}")

In [ ]:


class Config:
    MODEL_NAME = "vinai/phobert-large"
    TRAIN_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\train.csv"
    VAL_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\validation.csv"
    TEST_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\test.csv"
    MAX_LEN = 256
    NUM_LABELS = 3
    BATCH_SIZE = 12
    LR = 1.5e-5
    EPOCHS = 10
    PATIENCE = 3
    SEEDS = [42]
    NUM_EXPERTS = 8
    ROUTING_TYPES_TO_TEST = ["expert_choice", "smoe", "micro", "adaptive", "deepseek"]
    CAPACITY_FACTOR = 1.5

# ================= QUẢN LÝ THƯ MỤC THỰC NGHIỆM =================
Config.BASE_DIR = "experiments/PhoBERT_large"  # Đổi thành "experiments/XLM" hoặc "PhoBERT" tùy file
Config.RESUME_EXPERIMENT = True  # MỚI: Đặt True để chạy tiếp folder đang dang dở, False để tạo mới

os.makedirs(Config.BASE_DIR, exist_ok=True)
existing_exps = [d for d in os.listdir(Config.BASE_DIR) if d.startswith("experiment_")]
exp_nums = [int(d.split("_")[1]) for d in existing_exps if len(d.split("_")) > 1 and d.split("_")[1].isdigit()]

if Config.RESUME_EXPERIMENT and exp_nums:
    next_exp = max(exp_nums)
    print(f"🔄 CHẾ ĐỘ RESUME: Chạy tiếp tục tại phiên thực nghiệm {next_exp}")
else:
    next_exp = max(exp_nums) + 1 if exp_nums else 1
    print(f"📁 CHẾ ĐỘ NEW: Đã tạo phiên thực nghiệm mới experiment_{next_exp}")

Config.EXP_DIR = os.path.join(Config.BASE_DIR, f"experiment_{next_exp}")
os.makedirs(Config.EXP_DIR, exist_ok=True)

Config.CHECKPOINT_DIR = Config.EXP_DIR
Config.TRAIN_LOG_CSV = os.path.join(Config.EXP_DIR, "training_log.csv")
Config.RESULTS_CSV = os.path.join(Config.EXP_DIR, "result.csv")
Config.HYPERPARAMS_JSON = os.path.join(Config.EXP_DIR, "hyperparameters.json")

In [ ]:
label_map = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

class NLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.labels = torch.tensor([label_map.get(str(l).strip().lower(), 1) for l in df['label']], dtype=torch.long)
        print(f"Pre-tokenizing {len(df)} samples...")
        self.encodings = tokenizer(
            df['premise'].astype(str).tolist(),
            df['hypothesis'].astype(str).tolist(),
            max_length=max_len, padding='max_length', truncation=True, return_tensors='pt'
        )
        print(" Tokenization done!")

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

df_train = pd.read_csv(Config.TRAIN_CSV).dropna().reset_index(drop=True)
df_val = pd.read_csv(Config.VAL_CSV).dropna().reset_index(drop=True)
df_test = pd.read_csv(Config.TEST_CSV).dropna().reset_index(drop=True)

train_loader = DataLoader(NLIDataset(df_train, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(NLIDataset(df_val, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, pin_memory=True)
test_loader = DataLoader(NLIDataset(df_test, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, pin_memory=True)

In [ ]:
# ========================= UTILS & ARCHITECTURE =========================
class CheckpointManager:
    def __init__(self, model, optimizer, scheduler, scaler, model_name="moe"):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.scaler = scaler
        self.model_name = model_name
        self.best_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_best.pt")
        self.last_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_last.pt")
        self.train_log_path = Config.TRAIN_LOG_CSV
        
        if not os.path.exists(self.train_log_path):
            df = pd.DataFrame(columns=["Seed", "Epoch", "Routing", "Val_Acc", "Val_F1", "Val_Runtime_ms", "Val_VRAM_MB", "Val_Entropy", "Val_Expert_Usage"])
            df.to_csv(self.train_log_path, index=False)

    def save_checkpoint(self, epoch, val_f1, val_acc, is_best=False):
        state = {
            'epoch': epoch, 
            'model_state': self.model.state_dict(),
            'optimizer_state': self.optimizer.state_dict(),
            'scheduler_state': self.scheduler.state_dict(),
            'scaler_state': self.scaler.state_dict(),
            'best_val_f1': val_f1,
            'best_val_acc': val_acc
        }
        torch.save(state, self.last_checkpoint_path) # Luôn lưu mốc cuối cùng để Resume
        if is_best: 
            torch.save(state, self.best_checkpoint_path)

    def load_checkpoint(self):
        start_epoch, best_val_f1, best_val_acc = 0, 0.0, 0.0
        if os.path.exists(self.last_checkpoint_path):
            state = torch.load(self.last_checkpoint_path, map_location=device)
            # Dọn dẹp key rác của thop (nếu có)
            clean_state_dict = {k: v for k, v in state['model_state'].items() if 'total_ops' not in k and 'total_params' not in k}
            self.model.load_state_dict(clean_state_dict, strict=False)
            
            if 'optimizer_state' in state:
                self.optimizer.load_state_dict(state['optimizer_state'])
                self.scheduler.load_state_dict(state['scheduler_state'])
                self.scaler.load_state_dict(state['scaler_state'])
                
            start_epoch = state['epoch'] + 1
            best_val_f1 = state.get('best_val_f1', 0.0)
            best_val_acc = state.get('best_val_acc', 0.0)
            print(f"🔋 Đã khôi phục {self.model_name}! Chạy tiếp từ Epoch {start_epoch + 1}...")
        return start_epoch, best_val_f1, best_val_acc

    def log_training(self, row):
        df = pd.read_csv(self.train_log_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(self.train_log_path, index=False)

def get_backbone_info(model):
    """Tính toán số lượng tham số và bộ nhớ của riêng phần Backbone PLM"""
    param_count = sum(p.numel() for p in model.backbone.parameters())
    param_memory_mb = sum(p.nelement() * p.element_size() for p in model.backbone.parameters()) / (1024 * 1024)
    return param_count, param_memory_mb

def load_balancing_loss(router_output, num_experts):
    if router_output is None: return torch.tensor(0.0, device=device)
    if isinstance(router_output, tuple): router_output = router_output[0]
    probs = torch.softmax(router_output, dim=-1)
    expert_probs = probs.mean(dim=0)
    ideal = torch.ones(num_experts, device=probs.device) / num_experts
    return torch.mean((expert_probs - ideal) ** 2)

def calculate_routing_metrics(model):
    metrics = {"entropy": 0.0, "expert_usage_distribution": None}
    try:
        moe = model.moe_layer
        if hasattr(moe, "gate_logits") and moe.gate_logits is not None:
            probs = torch.softmax(moe.gate_logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean()
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = probs.mean(dim=0).cpu().numpy().tolist()
        elif hasattr(moe, "expert_usage") and moe.expert_usage is not None:
            usage = moe.expert_usage.float()
            probs = usage / (usage.sum() + 1e-9)
            entropy = -(probs * torch.log(probs + 1e-9)).sum()
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = usage.cpu().numpy().tolist()
    except Exception: pass
    return metrics

class LayerAttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(hidden_size, hidden_size), nn.Tanh(), nn.Linear(hidden_size, 1))

    def forward(self, hidden_states, attention_mask):
        attn_weights = self.attention(hidden_states).squeeze(-1)
        min_val = torch.finfo(attn_weights.dtype).min
        attn_weights = attn_weights.masked_fill(attention_mask == 0, min_val)
        weights = F.softmax(attn_weights, dim=-1)
        return torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)

class UnifiedMoENLI(nn.Module):
    def __init__(self, config, routing_type):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(config.MODEL_NAME)
        hidden_size = self.backbone.config.hidden_size
        self.routing_type = routing_type

        if routing_type == "expert_choice": self.moe_layer = ExpertChoiceMoELayer(hidden_size, config.NUM_EXPERTS, config.CAPACITY_FACTOR)
        elif routing_type == "smoe": self.moe_layer = SMoELayer(hidden_size, config.NUM_EXPERTS)
        elif routing_type == "micro": self.moe_layer = MICROMoELayer(hidden_size)
        elif routing_type == "adaptive": self.moe_layer = AdaptiveDynamicMoELayer(hidden_size, config.NUM_EXPERTS)
        elif routing_type == "deepseek": self.moe_layer = DeepSeekMoELayer(hidden_size, num_shared_experts=2, num_routed_experts=config.NUM_EXPERTS-2)
        else: self.moe_layer = ExpertChoiceMoELayer(hidden_size, config.NUM_EXPERTS, config.CAPACITY_FACTOR)

        self.pooling = LayerAttentionPooling(hidden_size)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(hidden_size, hidden_size // 2), nn.GELU(), nn.Linear(hidden_size // 2, config.NUM_LABELS))

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        moe_output = self.moe_layer(sequence_output)
        pooled = self.pooling(moe_output, attention_mask)
        return self.classifier(pooled)

In [ ]:
# ========================= MEGA PIPELINE =========================
for seed in Config.SEEDS:
    set_seed(seed)
    for routing_type in Config.ROUTING_TYPES_TO_TEST:
        print(f"\n{'='*70}\n🚀 SEED {seed} | ROUTING: {routing_type.upper()} | {Config.NUM_EXPERTS} Experts\n{'='*70}")

        model = UnifiedMoENLI(Config(), routing_type).to(device)
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=Config.LR, weight_decay=0.01)
        total_steps = len(train_loader) * Config.EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
        scaler = GradScaler()
        criterion = nn.CrossEntropyLoss()
        
        model_name = f"phobert_large_{routing_type}_seed{seed}"
        checkpoint_manager = CheckpointManager(model, model_name=model_name)

        dummy_ids = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
        dummy_mask = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
        macs, _ = profile(model, inputs=(dummy_ids, dummy_mask), verbose=False)
        gflops = (macs * 2) / 1e9

        best_val_f1 = 0.0
        best_val_acc = 0.0
        patience_counter = 0

        for epoch in range(Config.EPOCHS):
            model.train()
            train_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1} [{routing_type}]", leave=False)
            for batch in train_iterator:
                ids = batch['input_ids'].to(device, non_blocking=True)
                mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                with autocast():
                    logits = model(ids, mask)
                    ce_loss = criterion(logits, labels)
                    router_out = None
                    if hasattr(model.moe_layer, 'router'):
                        try: router_out = model.moe_layer.router(ids)
                        except: pass
                    lb_loss = load_balancing_loss(router_out, Config.NUM_EXPERTS)
                    loss = ce_loss + 0.01 * lb_loss

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                train_iterator.set_postfix(loss=f"{loss.item():.4f}")

            # --- VALIDATION ---
            model.eval()
            val_preds, val_labels = [], []
            torch.cuda.synchronize()
            start_time = time.time()
            
            with torch.inference_mode():
                for batch in val_loader:
                    ids = batch['input_ids'].to(device, non_blocking=True)
                    mask = batch['attention_mask'].to(device, non_blocking=True)
                    labels = batch['labels'].to(device, non_blocking=True)
                    with autocast():
                        logits = model(ids, mask)
                    val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                    val_labels.extend(labels.cpu().numpy())
                    
            torch.cuda.synchronize()
            runtime_ms = ((time.time() - start_time) / len(val_loader)) * 1000
            vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

            val_acc = accuracy_score(val_labels, val_preds)
            val_f1 = f1_score(val_labels, val_preds, average='macro')
            routing_stats = calculate_routing_metrics(model)

            print(f"Epoch {epoch+1} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | {runtime_ms:.2f} ms/b | VRAM: {vram_mb:.0f} MB")
            
            checkpoint_manager.log_training({
                "Seed": seed, "Epoch": epoch+1, "Routing": routing_type, 
                "Val_Acc": val_acc, "Val_F1": val_f1, "Val_Runtime_ms": runtime_ms, 
                "Val_VRAM_MB": vram_mb, "Val_Entropy": routing_stats["entropy"], "Val_Expert_Usage": str(routing_stats["expert_usage_distribution"])
            })

            is_best = val_f1 > best_val_f1
            if is_best:
                best_val_f1 = val_f1
                best_val_acc = val_acc
                patience_counter = 0
                print("✨ Val F1 cải thiện, lưu Best Checkpoint.")
            else:
                patience_counter += 1
                if patience_counter >= Config.PATIENCE:
                    print("🛑 Early stopping kích hoạt!")
                    break
                    
            checkpoint_manager.save_checkpoint(epoch, best_val_f1, is_best)

        # ================= TEST EVALUATION (MỚI: ĐẦY ĐỦ METRICS) =================
        print(f"\n📥 Loading best checkpoint cho {routing_type.upper()}...")
        state = torch.load(checkpoint_manager.best_checkpoint_path, map_location=device)
        model.load_state_dict(state["model_state"])
        model.eval()
        
        test_preds, test_labels = [], []
        torch.cuda.reset_peak_memory_stats() # Reset để đo lại VRAM chuẩn của Test
        torch.cuda.synchronize()
        test_start_time = time.time()
        
        with torch.inference_mode():
            for batch in test_loader:
                ids = batch['input_ids'].to(device, non_blocking=True)
                mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)
                with autocast():
                    logits = model(ids, mask)
                test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                test_labels.extend(labels.cpu().numpy())

        torch.cuda.synchronize()
        test_runtime_ms = ((time.time() - test_start_time) / len(test_loader)) * 1000
        test_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

        test_acc = accuracy_score(test_labels, test_preds)
        test_f1 = f1_score(test_labels, test_preds, average='macro')
        
        test_routing_stats = calculate_routing_metrics(model)
        param_count, param_memory_mb = get_backbone_info(model)
        
        print(f"🎯 TEST ACC = {test_acc:.4f} | TEST F1 = {test_f1:.4f}\n")

        # Lưu gộp tất cả thông tin quan trọng vào RESULT.CSV của experiment
        res_df = pd.DataFrame([{
            "Seed": seed, "Routing": routing_type,
            "Best_Val_Acc": best_val_acc, "Best_Val_F1": best_val_f1,
            "Test_Acc": test_acc, "Test_F1": test_f1, "GFLOPS": gflops, 
            "Test_Runtime_ms": test_runtime_ms, "Test_VRAM_MB": test_vram_mb,
            "Test_Entropy": test_routing_stats["entropy"], 
            "Test_Expert_Usage": str(test_routing_stats["expert_usage_distribution"]),
            "Backbone_Params": param_count, "Backbone_Memory_MB": param_memory_mb
        }])
        
        if not os.path.exists(Config.RESULTS_CSV):
            res_df.to_csv(Config.RESULTS_CSV, index=False)
        else:
            res_df.to_csv(Config.RESULTS_CSV, mode='a', header=False, index=False)

        del model, optimizer, scheduler, checkpoint_manager
        torch.cuda.empty_cache()
        gc.collect()

print(f"✅ Hoàn tất! Báo cáo Test và Train Log đã được lưu tại thư mục: {Config.EXP_DIR}")